In [ ]:
import numpy as np
import warnings
import csv  # для сохранения в CSV

warnings.filterwarnings('ignore')

# Создаём структурированный dtype с типами полей согласно варианту
dtype = np.dtype([('ts', 'int32'), ('lab_id', 'int16'),
                  ('hgb', 'float32'), ('glu', 'float32'),
                  ('wbc', 'float32'), ('plt', 'float32')])

print("Загрузка data.csv...")
# np.genfromtxt - читает CSV файл, пропуская заголовок (skip_header=1)
data = np.genfromtxt('data.csv', delimiter=',', skip_header=1, dtype=dtype, encoding='utf-8')
print(f"Строк: {len(data)}")
print(f"Память: {data.nbytes} байт ({data.nbytes / 1024**2:.2f} МБ)")

# Подсчёт NaN и Inf в числовых полях
for field in ['hgb', 'glu', 'wbc', 'plt']:
    col = data[field]
    bad = np.isnan(col).sum() + np.isinf(col).sum()  # сумма пропусков и бесконечностей
    pct = bad / len(data) * 100
    print(f"{field}: {bad} ({pct:.2f}%)")
    if pct > 3:
        print(f"  Воу, воу. Уже >3% в {field}")

with open('data_original.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['ts', 'lab_id', 'hgb', 'glu', 'wbc', 'plt'])  # заголовки
    for row in data:
        writer.writerow([row['ts'], row['lab_id'], row['hgb'], row['glu'], row['wbc'], row['plt']])
print("Сохранено data_original.csv\n")

In [ ]:
data_clean = data.copy()

# np.clip - ограничивает значения заданными пределами (здесь [20, 500])
glu_before = data_clean['glu']
data_clean['glu'] = np.clip(glu_before, 20, 500)
print(f"Глюкоза обрезана: {np.sum((glu_before<20)|(glu_before>500))} аномалий")

# np.where - условная замена: если wbc < 0, то 0, иначе оставляем как есть
wbc_before = data_clean['wbc']
data_clean['wbc'] = np.where(wbc_before < 0, 0, wbc_before)
print(f"Лейкоциты: {np.sum(wbc_before < 0)} отрицательных -> 0")

# Вычисляем 95-й процентиль, игнорируя NaN
hgb_95 = np.nanpercentile(data_clean['hgb'], 95)

# Проверяем, удалось ли вычислить процентиль
if np.isnan(hgb_95):
    print("ОШИБКА: слишком много пропусков в гемоглобине, используем медиану")
    hgb_95 = np.nanmedian(data_clean['hgb'])
    if np.isnan(hgb_95):
        print("КРИТИЧЕСКАЯ ОШИБКА: все значения гемоглобина NaN, используем норму 140")
        hgb_95 = 140.0
    else:
        print(f"   Медиана гемоглобина = {hgb_95:.2f} г/л (используем как границу)")
else:
    print(f"   95-й процентиль гемоглобина = {hgb_95:.2f} г/л")
    print(f"   Это означает, что 5% пациентов имеют гемоглобин выше {hgb_95:.2f} г/л")

# Ограничиваем сверху 95-м процентилем
data_clean['hgb'] = np.clip(data_clean['hgb'], None, hgb_95)

# Подсчитываем, сколько значений было обрезано
original_hgb = data['hgb']
clipped_count = np.sum((original_hgb > hgb_95) & ~np.isnan(original_hgb))
print(f"   Обрезано {clipped_count} значений (выше {hgb_95:.2f} г/л)\n")

In [ ]:
# np.unique - возвращает уникальные значения lab_id
unique_labs = np.unique(data_clean['lab_id'])
print(f"Количество лабораторий: {len(unique_labs)}")

# Расширяем dtype новым полем glu_zscore
new_dtype = np.dtype(data_clean.dtype.descr + [('glu_zscore', 'float32')])
data_norm = np.zeros(len(data_clean), dtype=new_dtype)
for f in data_clean.dtype.names:
    data_norm[f] = data_clean[f]

sizes = []
for lab in unique_labs:
    mask = data_norm['lab_id'] == lab  # булева маска для текущей группы
    glu_group = data_norm['glu'][mask]
    sizes.append(len(glu_group))
    mean_g = np.nanmean(glu_group)  # среднее с игнорированием NaN
    std_g = np.nanstd(glu_group)    # стандартное отклонение с игнорированием NaN
    # Z-score: (x - mean) / std, если std != 0, иначе 0
    data_norm['glu_zscore'][mask] = (glu_group - mean_g) / std_g if std_g != 0 else 0.0
print(f"Размеры групп: мин {min(sizes)}, макс {max(sizes)}, среднее {np.mean(sizes):.1f}")

# Условная агрегация для анемичных пациентов (hgb < 120)
condition = data_norm['hgb'] < 120
agg_results = []
for lab in unique_labs:
    mask = (data_norm['lab_id'] == lab) & condition  # составная маска
    sub_glu = data_norm['glu'][mask]
    if len(sub_glu) > 0:
        # np.nanmean, np.nanmedian, np.nanpercentile - игнорируют NaN
        agg_results.append((lab, np.nanmean(sub_glu), np.nanmedian(sub_glu), np.nanpercentile(sub_glu, 90)))
    else:
        agg_results.append((lab, np.nan, np.nan, np.nan))
agg_table = np.array(agg_results, dtype=[('lab_id','i2'),('mean_cond','f4'),('median_cond','f4'),('p90_cond','f4')])
print("\nУсловная агрегация (анемия) — первые 5 строк:")
print(agg_table[:5])

# Сохранение в CSV
with open('data_normalized.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['ts', 'lab_id', 'hgb', 'glu', 'wbc', 'plt', 'glu_zscore'])
    for i in range(len(data_norm)):
        writer.writerow([data_norm['ts'][i], data_norm['lab_id'][i], data_norm['hgb'][i],
                        data_norm['glu'][i], data_norm['wbc'][i], data_norm['plt'][i],
                        data_norm['glu_zscore'][i]])
print("Сохранено data_normalized.csv")

In [ ]:
k = 30
glu_vals = data_norm['glu']
# np.nan_to_num - заменяет NaN на 0 для корректной работы cumsum
glu_filled = np.nan_to_num(glu_vals, nan=0.0)
cumsum_filled = np.cumsum(glu_filled)  # кумулятивная сумма
ma30 = np.full(len(glu_vals), np.nan, dtype=np.float32)
# Формула скользящего среднего через кумулятивную сумму: O(N) без циклов
ma30[k-1:] = (cumsum_filled[k-1:] - np.concatenate(([0], cumsum_filled[:-k]))) / k

hgb_vals = data_norm['hgb'].copy()
if np.isnan(hgb_vals).any():
    print("ВНИМАНИЕ: обнаружены NaN в hgb, заменяем на медиану")
    median_hgb = np.nanmedian(hgb_vals)  # медиана с игнорированием NaN
    hgb_vals = np.nan_to_num(hgb_vals, nan=median_hgb)

# np.diff - вычисляет разности между соседними элементами
# Реализуем вручную, так как первый элемент должен быть NaN
hgb_diff = np.empty_like(hgb_vals, dtype=np.float32)
hgb_diff[0] = np.nan
hgb_diff[1:] = hgb_vals[1:] - hgb_vals[:-1]

# np.roll - циклический сдвиг массива (для создания лагового признака)
hgb_lag1 = np.roll(hgb_vals, 1)
hgb_lag1[0] = np.nan  # граничный элемент обрабатываем как NaN
diff_lag = hgb_vals - hgb_lag1
valid = ~np.isnan(diff_lag)
if valid.sum() == 0:
    print("Нет данных для расчёта изменений гемоглобина.")
    counts = np.array([0, 0, 0])
else:
    signs = np.sign(diff_lag[valid])  # np.sign возвращает -1, 0 или 1
    # np.bincount считает частоты, +1 смещает индекс (-1->0, 0->1, 1->2)
    counts = np.bincount(signs.astype(int) + 1)

print("Распределение изменений гемоглобина (от записи к записи):")
print(f"  Упало  : {counts[0]} ({counts[0]/len(valid)*100 if valid.sum()>0 else 0:.2f}%)")
print(f"  Не изм.: {counts[1]} ({counts[1]/len(valid)*100 if valid.sum()>0 else 0:.2f}%)")
print(f"  Выросло: {counts[2]} ({counts[2]/len(valid)*100 if valid.sum()>0 else 0:.2f}%)")

# Расширяем массив новыми полями
new_dtype2 = np.dtype(data_norm.dtype.descr + [('glu_ma30', 'f4'), ('hgb_diff', 'f4'), ('hgb_lag1', 'f4')])
data_ext = np.zeros(len(data_norm), dtype=new_dtype2)
for f in data_norm.dtype.names:
    data_ext[f] = data_norm[f]
data_ext['glu_ma30'] = ma30
data_ext['hgb_diff'] = hgb_diff
data_ext['hgb_lag1'] = hgb_lag1
data_norm = data_ext
print("\nДобавлены: скользящее среднее глюкозы, diff гемоглобина, лаг 1 гемоглобина")

In [ ]:
# np.errstate - подавляет предупреждения о делении на ноль
with np.errstate(divide='ignore', invalid='ignore'):
    metab = np.where(data_norm['hgb'] != 0, data_norm['glu'] / data_norm['hgb'], 0.0)
metab = np.nan_to_num(metab, nan=0.0, posinf=0.0, neginf=0.0)  # замена Inf/NaN на 0

plt_wbc = data_norm['plt'] * data_norm['wbc']
plt_wbc = np.nan_to_num(plt_wbc, nan=0.0)

new_dtype3 = np.dtype(data_norm.dtype.descr + [('metab_index', 'f4'), ('plt_wbc', 'f4')])
data_feat = np.zeros(len(data_norm), dtype=new_dtype3)
for f in data_norm.dtype.names:
    data_feat[f] = data_norm[f]
data_feat['metab_index'] = metab.astype(np.float32)
data_feat['plt_wbc'] = plt_wbc.astype(np.float32)
data_norm = data_feat
print("Производные признаки добавлены.")

# Групповая замена выбросов по IQR
total_replaced = 0
for lab in unique_labs:
    mask = data_norm['lab_id'] == lab
    group = data_norm['glu'][mask]
    if len(group) == 0:
        continue
    # np.nanpercentile - процентиль с игнорированием NaN
    q1, q3 = np.nanpercentile(group, [25, 75])
    iqr = q3 - q1  # межквартильный размах
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr  # границы для выбросов
    outliers = (group < lower) | (group > upper)
    n_out = np.sum(outliers)
    if n_out:
        median = np.nanmedian(group)  # медиана для замены
        data_norm['glu'][mask][outliers] = median
        total_replaced += n_out
        print(f"Лаб. {lab}: заменено {n_out} выбросов из {len(group)}")
print(f"\nВсего замен: {total_replaced} ({total_replaced/len(data_norm)*100:.3f}%)")

In [ ]:
# Составная булева маска для проверки логических правил
violation = (data_norm['hgb'] < 50) | (data_norm['hgb'] > 200) | \
            (data_norm['glu'] < 20) | (data_norm['glu'] > 500) | \
            (data_norm['wbc'] < 0) | (data_norm['plt'] < 0)
print(f"Нарушений логики: {np.sum(violation)} ({np.sum(violation)/len(data_norm)*100:.2f}%)")

# np.where - условная замена значений
new_dtype4 = np.dtype(data_norm.dtype.descr + [('anomaly_flag', 'int8')])
data_final = np.zeros(len(data_norm), dtype=new_dtype4)
for f in data_norm.dtype.names:
    data_final[f] = data_norm[f]
data_final['anomaly_flag'] = 0
data_final['anomaly_flag'][violation] = 1
data_final['anomaly_flag'] = np.where(violation, 0, data_final['anomaly_flag'])
print("Флаг аномалии приведён к эталону (0).")

# np.isin - проверяет вхождение элементов в заданный список
lab_ids = data_final['lab_id']
unique, counts = np.unique(lab_ids, return_counts=True)
rare = unique[counts < 0.01 * len(data_final)]  # категории < 1%
print(f"Редких лабораторий (<1%): {len(rare)} из {len(unique)}")

# Создаём сжатое поле для lab_id (редкие -> 0)
if 'lab_id_compressed' not in data_final.dtype.names:
    new_dtype5 = np.dtype(data_final.dtype.descr + [('lab_id_compressed', 'int16')])
    data_comp = np.zeros(len(data_final), dtype=new_dtype5)
    for f in data_final.dtype.names:
        data_comp[f] = data_final[f]
    data_final = data_comp
data_final['lab_id_compressed'] = data_final['lab_id']
data_final['lab_id_compressed'][np.isin(data_final['lab_id'], rare)] = 0  # OTHER = 0
print("Редкие категории заменены на 0 (OTHER).")

# Сохранение финального массива в CSV
with open('data_final.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['ts', 'lab_id', 'hgb', 'glu', 'wbc', 'plt', 'glu_zscore',
                     'glu_ma30', 'hgb_diff', 'hgb_lag1', 'metab_index', 'plt_wbc',
                     'anomaly_flag', 'lab_id_compressed'])
    for i in range(len(data_final)):
        writer.writerow([data_final['ts'][i], data_final['lab_id'][i], data_final['hgb'][i],
                        data_final['glu'][i], data_final['wbc'][i], data_final['plt'][i],
                        data_final['glu_zscore'][i], data_final['glu_ma30'][i],
                        data_final['hgb_diff'][i], data_final['hgb_lag1'][i],
                        data_final['metab_index'][i], data_final['plt_wbc'][i],
                        data_final['anomaly_flag'][i], data_final['lab_id_compressed'][i]])

print("\nФинальный массив сохранён как 'data_final.csv'")
print(f"Итоговый размер: {len(data_final)} строк, поля: {data_final.dtype.names}")